# 🚋 System Optymalizacji Rozkładu Jazdy Tramwajów
Witaj w aplikacji wspierającej decyzje dyspozytorskie. Ten interaktywny notatnik przeprowadzi Cię przez dwufazowy proces tworzenia optymalnego rozkładu jazdy. 

**Jak korzystać z aplikacji?**
Zaznaczaj kolejne komórki i uruchamiaj je za pomocą przycisku `Run` (lub skrótu `Shift + Enter`). Czytaj uważnie opisy przed każdym krokiem, aby w pełni kontrolować parametry algorytmu.

---
## KROK 1: Przygotowanie Danych Wejściowych
W pierwszej kolejności system musi zrozumieć, jak wygląda miasto. W tym celu ładujemy z folderu `data/input/` odpowiednie pliki `.csv`, które algorytm zamienia na szybkie macierze matematyczne:
* **Macierz Popytu ($P$)** – określa, ilu pasażerów chce przemieścić się między poszczególnymi przystankami w danej godzinie.
* **Macierz Czasów ($W$)** – przechowuje czasy przejazdu (w minutach) pomiędzy węzłami sieci.

In [15]:
from data_generator import generate_dummy_data
from data_loader import load_data_as_matrices

num_stops = 5
num_hours = 24
generate_dummy_data(num_stops, num_hours)
P,W = load_data_as_matrices("data/input/travel_times.csv", "data/input/demand.csv", num_stops, num_hours)

✅ Pomyślnie wygenerowano dane odnośnie czasów przejazdu oraz popytu
✅ Pomyślnie załadowano dane wejściowe w odpowieniej formie numpy


---
## KROK 2: Konfiguracja Silnika Optymalizacji (Metaheurystyka)
Nasz system wykorzystuje algorytm **Symulowanego Wyżarzania (Simulated Annealing)**, inspirowany fizycznym procesem stygnięcia metali. W poniższej komórce decydujesz, jak dokładnie system ma przeszukiwać warianty rozkładów:

* `initial_temp` / `min_temp` – Temperatura początkowa i końcowa. Im wyższa temperatura początkowa, tym chętniej algorytm "bada" nietypowe i gorsze rozwiązania, aby uciec z minimów lokalnych.
* `cooling_rate` – Szybkość stygnięcia (np. 0.999). Wartości bliższe 1.0 dają lepsze wyniki (dokładniejsza eksploracja), ale wydłużają czas obliczeń.
* `max_frequency` – Zabezpieczenie przed przepełnieniem infrastruktury (tzw. wąskie gardło). Oznacza maksymalną dopuszczalną liczbę kursów na godzinę, jakiej tory i perony są w stanie sprostać fizycznie.

In [10]:
initial_temp = 5.0
min_temp = 1e-3
cooling_rate=0.999
max_iter=1000
log_every=250
max_frequency=30

---
## KROK 3: Parametry Ekonomiczno-Fizyczne i Faza 1
Zanim uruchomimy obliczenia, musimy zdefiniować twarde warunki operacyjne oraz ustalić, na czym najbardziej zależy przewoźnikowi. Służą do tego 4 kluczowe parametry:

**Parametry Fizyczne:**
* **$C$ (Pojemność taboru)** – Ile osób maksymalnie zmieści się w jednym pojeździe (wymusza odpowiednią częstotliwość w godzinach szczytu).
* **$R$ (Czas obrotu)** – Ile minut zajmuje tramwajowi pełny kurs w tę i z powrotem. Pozwala przeliczyć kursy na fizyczną wielkość niezbędnej floty ($V$).

**Parametry Ekonomiczne (Wagi funkcji celu):**
* **$\alpha$ (Alfa)** – Priorytet pasażera. Im wyższy, tym system będzie bardziej "bał się" kar za oczekiwanie ludzi na przystankach i wypuści więcej pojazdów.
* **$\beta$ (Beta)** – Priorytet kosztowy. Im wyższy, tym system będzie bardziej oszczędzał na flocie, minimalizując liczbę wypuszczanych w miasto tramwajów.

*Po uruchomieniu poniższej komórki rozpocznie się Faza 1: Wyznaczanie optymalnej liczby kursów dla każdej godziny.*

In [11]:
C=30
R=30
ALPHA=1.0
BETA=30.0

In [12]:
from annealing_helpers import build_problem_data, run_schedule_optimization, print_results

data = build_problem_data(P, C=C, R=R, alpha=ALPHA, beta=BETA)

results = run_schedule_optimization(
    data,
    initial_temp=initial_temp,
    min_temp=min_temp,
    cooling_rate=cooling_rate,
    max_iter=max_iter,
    log_every=log_every,
    max_frequency=max_frequency
)
#print_results(results)

---
## KROK 4: Faza 2 – Ciągłe Mikro-Harmonogramowanie
Znamy już idealną liczbę kursów na każdą godzinę, ale system musi przekuć to w faktyczny rozkład jazdy (konkretne minuty odjazdów).

Naiwne podejście (podział 60 minut przez liczbę kursów) spowodowałoby ogromne, niefizyczne luki na stykach godzin (np. przeskok z tramwaju co 10 minut na tramwaj co 30 minut). Nasz autorski system wykorzystuje mapowanie poprzez **Krzywą Skumulowaną (CDF)**. Algorytm w naturalny sposób wygładza odstępy między kursami, zachowując płynność i minimalizując szok dla pasażerów przy zmianie natężenia ruchu.

In [13]:
from timetable_phase import generate_timetable

timetable_df = generate_timetable(results=results, W_matrix=W)
final_timetable_path = "data/output/final_timetable.csv"
timetable_df.to_csv(final_timetable_path, index=False)

---
## KROK 5: Zapis Danych i Wygenerowanie Plakatów PDF
Gotowy, gładki rozkład jest teraz w pamięci systemu. W tym kroku aplikacja:
1. Zapisze surowe dane do pliku tekstowego `final_timetable.csv` w celach archiwizacji.
2. Odczyta przyjazne nazwy przystanków z Twojego pliku `stop_names.csv`.
3. Wygeneruje dla każdego przystanku gotowy do druku, profesjonalny plakat PDF, który automatycznie trafi do folderu `data/output/pdf`.

*Zajrzyj do folderu docelowego po zakończeniu obliczeń, aby pobrać gotowe materiały.*

In [14]:
from pdf_generator import generate_pdf_posters
from data_loader import load_stop_names

stop_names = load_stop_names("data/input/stop_names.csv", num_stops=num_stops)
generate_pdf_posters(final_timetable_path, stop_names_dictionary=stop_names, output_dir="data/output/pdf")

✅ Pomyślnie załadowano nazwy przystanków
📖 Wczytywanie rozkładu jazdy z pliku: data/output/final_timetable.csv...
✅ Sukces! Wygenerowano 5 plakatów PDF w folderze data/output/pdf/
